# MODEL COMPARISON

In [ ]:
from pathlib import Path
import numpy as np
import csv
import torch
from torchvision.transforms import ToTensor
from PIL import Image
from typing import Any

from utils.exr import read_exr_red
from utils.legacy_depth_correction import backproject_z, correct_legacy_depth

from models.model import GLPDepth
from types import SimpleNamespace
from collections import OrderedDict

import utils.logging_ as logging
import utils.metrics as metrics

import matplotlib.pyplot as plt
%matplotlib ipympl

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

In [ ]:
metric_name = ['d1', 'd2', 'd3', 'abs_rel', 'sq_rel', 'rmse', 'rmse_log', 'log10', 'silog']
result_metrics = {}
for metric in metric_name:
    result_metrics[metric] = 0.0

In [ ]:
def show_depth(depth, normalization=None):
    depth_img = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    depth_img = (depth_img * 255).astype(np.uint8)

    print("DEPTH MIN:", depth.min())
    print("DEPTH MAX:", depth.max())
    print("NORMALIZATION:", normalization)

    Image.fromarray(depth_img).show()

def valid_depth_mask(depth):
    """Pixels that contain a finite, positive depth value."""
    depth = np.asarray(depth)
    return np.isfinite(depth) & (depth > 0)


def shared_depth_bounds(depth_maps, valid):
    """Return the global valid min/max across a mapping of depth maps."""
    ranges = {}
    for name, depth_map in depth_maps.items():
        mask = valid & np.isfinite(depth_map)
        if not np.any(mask):
            raise ValueError(f"{name!r} contains no valid depth values.")
        values = np.asarray(depth_map)[mask]
        ranges[name] = (float(values.min()), float(values.max()))

    vmin = min(lo for lo, _ in ranges.values())
    vmax = max(hi for _, hi in ranges.values())
    return vmin, vmax, ranges


def plot_depth_comparison(depth_maps, valid, vmin=None, vmax=None, cmap="viridis", ncols=2):
    """Display all depth maps with one shared color normalization."""
    if vmin is None or vmax is None:
        vmin, vmax, _ = shared_depth_bounds(depth_maps, valid)

    n = len(depth_maps)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(6 * ncols, 4.5 * nrows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad("black")
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    image = None
    for ax, (name, depth_map) in zip(axes, depth_maps.items()):
        mask = valid & np.isfinite(depth_map)
        shown = np.ma.masked_where(~mask, depth_map)

        image = ax.imshow(shown, cmap=cmap_obj, norm=norm)
        ax.set_title(name)
        ax.set_axis_off()

    for ax in axes[n:]:
        ax.set_visible(False)

    cbar = fig.colorbar(image, ax=axes[:n], shrink=0.9, pad=0.02)
    cbar.set_label("Depth")
    fig.suptitle(
        f"Depth maps — shared valid range [{vmin:.4f}, {vmax:.4f}]",
        fontsize=14,
    )
    plt.show()


def _pointcloud_from_depth(depth, rgb, valid):
    """Backproject all finite positive depth pixels."""
    points_grid = backproject_z(depth)
    mask = valid & np.all(np.isfinite(points_grid), axis=-1)

    points = points_grid[mask]
    colors = rgb[mask] / 255.0

    if len(points) == 0:
        raise ValueError("Depth map produced an empty point cloud.")

    return points, colors


def _shared_pointcloud_limits(pointclouds, padding=0.05):
    """Return one cubic XYZ bounding box shared by every point cloud."""
    mins = np.stack([points.min(axis=0) for points, _ in pointclouds.values()])
    maxs = np.stack([points.max(axis=0) for points, _ in pointclouds.values()])

    xyz_min = mins.min(axis=0)
    xyz_max = maxs.max(axis=0)
    center = (xyz_min + xyz_max) / 2

    radius = max(np.ptp(np.stack([xyz_min, xyz_max]), axis=0).max() / 2, 1e-6)
    radius *= 1 + padding

    return (
        (center[0] - radius, center[0] + radius),
        (center[1] - radius, center[1] + radius),
        (center[2] - radius, center[2] + radius),
    )


def plot_pointcloud_comparison(
    depth_maps,
    rgb,
    valid,
    max_points=20_000,
    seed=7,
    ncols=2,
):
    """Display all point clouds with the same XYZ limits and camera view."""
    pointclouds = {
        name: _pointcloud_from_depth(depth_map, rgb, valid)
        for name, depth_map in depth_maps.items()
    }
    xlim, ylim, zlim = _shared_pointcloud_limits(pointclouds)

    n = len(pointclouds)
    nrows = int(np.ceil(n / ncols))
    fig = plt.figure(figsize=(7 * ncols, 6 * nrows))

    rng = np.random.default_rng(seed)

    for idx, (name, (points, colors)) in enumerate(pointclouds.items(), start=1):
        if len(points) > max_points:
            sample_idx = rng.choice(len(points), max_points, replace=False)
            points_to_plot = points[sample_idx]
            colors_to_plot = colors[sample_idx]
        else:
            points_to_plot = points
            colors_to_plot = colors

        ax = fig.add_subplot(nrows, ncols, idx, projection="3d")
        ax.scatter(
            *points_to_plot.T,
            c=colors_to_plot,
            s=3,
            linewidths=0,
            depthshade=False,
        )
        ax.set(
            xlabel="X",
            ylabel="Y",
            zlabel="Z",
            title=f"{name} ({len(points):,} valid points)",
            xlim=xlim,
            ylim=ylim,
            zlim=zlim,
        )
        ax.set_box_aspect((1, 1, 1))
        ax.invert_yaxis()
        ax.view_init(elev=-130, azim=90, roll=180)

    fig.suptitle("Point clouds — shared XYZ bounds", fontsize=14)
    plt.tight_layout()
    plt.show()


## > SAMPLE SELECTION

In [ ]:
dataset_root = None  # e.g. Path("/path/to/SHAPENETSEM")
dataset_metadata = None  # e.g. Path("/path/to/SHAPENETSEM-METADATA")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the SHAPENETSEM dataset.")

if dataset_metadata is None:
    raise ValueError("Please set `dataset_metada` to the path of the SHAPENETSEM dataset metadata file.")

dataset_root = Path(dataset_root)
dataset_metadata = Path(dataset_metadata)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")
    
if not dataset_metadata.exists():
    raise FileNotFoundError(f"Metadata path does not exist: {dataset_metadata}")

In [ ]:
sample_id = None
view = None

sample_id = "1da9362bcaa79e65136d9cfd13835101"
view = "top"

if sample_id is None or view is None:
    raise ValueError("Please set `sample_id` and `view` for the desired sample.")

sample_path = Path(dataset_root / sample_id / (str(sample_id) + "_" + str(view)))

if not Path(str(sample_path) + ".png").exists():
    raise FileNotFoundError(f"Sample path does not exist: {sample_path}")

In [ ]:
depth = read_exr_red(str(sample_path) + "_depth0001.exr")
depth_z = correct_legacy_depth(depth)
rgb = np.asarray(Image.open(str(sample_path) + ".png").convert("RGB"))

with dataset_metadata.open("r", encoding="utf-8", newline="") as f:
    row = next(
        (r for r in csv.DictReader(f) if r["fullId"] == f"wss.{sample_id}"),
        None,
    )
if row is None or not row["aligned.dims"]:
    raise ValueError(f"No aligned dimensions found for sample {sample_id}")

aligned_dims = np.array([float(x) for x in row["aligned.dims"].split(r"\,")]) / 100
normalization = np.linalg.norm(aligned_dims)

In [ ]:
Image.fromarray(rgb).show()

## > GLP-DEPTH PREDICTION

In [ ]:
to_tensor = ToTensor()
input_RGB = to_tensor(rgb).unsqueeze(0).to(device)

### >> ORIGINAL

In [ ]:
glp_ckpt_original = None  # e.g. Path("/path/to/GLP-CKPT-OG")

if glp_ckpt_original is None:
    raise ValueError("Please set `glp_ckpt_original` to the path of the original GLP-Depth checkpoint.")

glp_ckpt_original = Path(glp_ckpt_original)

if not glp_ckpt_original.exists():
    raise FileNotFoundError(f"Checkpoint path does not exist: {glp_ckpt_original}")

In [ ]:
# Model Arguments
args_original = SimpleNamespace(min_depth_eval=1e-3, max_depth_eval=2.63, max_depth=2.63)

In [ ]:
model_weight_original = torch.load(glp_ckpt_original)
if 'module' in next(iter(model_weight_original.items()))[0]:
    model_weight_original = OrderedDict((k[7:], v) for k, v in model_weight_original.items())

model_original = GLPDepth(max_depth=args_original.max_depth, is_train=False).to(device)
model_original.load_state_dict(model_weight_original)
model_original.eval()

In [ ]:
with torch.no_grad():
    pred = model_original(input_RGB)
pred_d_original_tensor = pred['pred_d'].squeeze() * normalization
pred_d_original = pred_d_original_tensor.detach().cpu().numpy().squeeze()


In [ ]:
show_depth(pred_d_original)

In [ ]:
# Evaluate Prediction Against Ground Truth
depth_gt = to_tensor(depth).to(device)
depth_gt = depth_gt.squeeze()
pred_crop, gt_crop = metrics.cropping_img(args_original, pred_d_original_tensor, depth_gt)
computed_result = metrics.eval_depth(pred_crop, gt_crop)

for metric in metric_name:
    result_metrics[metric] = computed_result[metric]

display_result = logging.display_result(result_metrics)
print(display_result)

### >> CORRECTED

In [ ]:
glp_ckpt_corrected = None  # e.g. Path("/path/to/GLP-CKPT")

if glp_ckpt_corrected is None:
    raise ValueError("Please set `glp_ckpt_corrected` to the path of the corrected GLP-Depth checkpoint.")

glp_ckpt_corrected = Path(glp_ckpt_corrected)

if not glp_ckpt_corrected.exists():
    raise FileNotFoundError(f"Checkpoint path does not exist: {glp_ckpt_corrected}")

In [ ]:
# Model Arguments
args_corrected = SimpleNamespace(min_depth_eval=1e-3, max_depth_eval=2.63, max_depth=2.63)

In [ ]:
model_weight_corrected = torch.load(glp_ckpt_corrected)
if 'module' in next(iter(model_weight_corrected.items()))[0]:
    model_weight_corrected = OrderedDict((k[7:], v) for k, v in model_weight_corrected.items())

model_corrected = GLPDepth(max_depth=args_corrected.max_depth, is_train=False).to(device)
model_corrected.load_state_dict(model_weight_corrected)
model_corrected.eval()

In [ ]:
with torch.no_grad():
    pred = model_corrected(input_RGB)
pred_d_corrected_tensor = pred['pred_d'].squeeze() * normalization
pred_d_corrected = pred_d_corrected_tensor.detach().cpu().numpy().squeeze()

In [ ]:
show_depth(pred_d_corrected)

In [ ]:
# Evaluate Prediction Against Ground Truth
depth_gt = to_tensor(depth_z).to(device)
depth_gt = depth_gt.squeeze()
pred_crop, gt_crop = metrics.cropping_img(args_corrected, pred_d_corrected_tensor, depth_gt)
computed_result = metrics.eval_depth(pred_crop, gt_crop)

for metric in metric_name:
    result_metrics[metric] = computed_result[metric]

display_result = logging.display_result(result_metrics)
print(display_result)

## > COMPARISON


In [ ]:
valid = depth > 0

depth_maps = {
    "Ground Truth | Original": depth,
    "Ground Truth | Corrected": depth_z,
    "Prediction | Original": pred_d_original,
    "Prediction | Corrected": pred_d_corrected,
}

depth_vmin, depth_vmax, depth_ranges = shared_depth_bounds(depth_maps, valid)

print("Valid depth range per map:")
for name, (lo, hi) in depth_ranges.items():
    print(f"  {name:28s} [{lo:.6f}, {hi:.6f}]")
print(f"\nShared display range: [{depth_vmin:.6f}, {depth_vmax:.6f}]")

plot_depth_comparison(
    depth_maps,
    valid,
    vmin=depth_vmin,
    vmax=depth_vmax,
    cmap="viridis",
    ncols=2,
)

plot_pointcloud_comparison(
    depth_maps,
    rgb,
    valid,
    max_points=20_000,
    seed=7,
    ncols=2,
)
